# **Montagem do Google Drive, Preparação dos Dados (Limpeza e Estruturação) e Média Ponderada**

In [1]:
import pandas as pd
import os
from google.colab import drive
import numpy as np

# ====================================================================
# 1. Montagem e Carregamento Reforçado
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Apenas monta se ainda não estiver montado
    drive.mount('/content/drive')
else:
    print("Drive já montado.")

# --- Caminho Corrigido ---
PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'
CAMINHO_SAVE_MATA = os.path.join(PASTA_DRIVE, 'Mata_estruturado.csv')
CAMINHO_SAVE_CASA = os.path.join(PASTA_DRIVE, 'Casa_estruturado.csv')

try:
    # Carregar df_mata com tratamento de erro de tipo
    df_mata = pd.read_csv(CAMINHO_SAVE_MATA, index_col=0, parse_dates=True)
    df_mata['Temperatura'] = pd.to_numeric(df_mata['Temperatura'], errors='coerce')
    df_mata['Umidade'] = pd.to_numeric(df_mata['Umidade'], errors='coerce')
    df_mata.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    # Carregar df_casa com tratamento de erro de tipo
    df_casa = pd.read_csv(CAMINHO_SAVE_CASA, index_col=0, parse_dates=True)
    # Garante que as colunas numéricas estejam como float, transformando erros em NaN
    df_casa['Temp'] = pd.to_numeric(df_casa['Temp'], errors='coerce')
    df_casa['Umid'] = pd.to_numeric(df_casa['Umid'], errors='coerce')
    df_casa.dropna(subset=['Temp', 'Umid'], inplace=True)

    print("DataFrames carregados e tipos de dados verificados com sucesso.")

except FileNotFoundError:
    print(f"\nERRO FATAL: Não foi possível encontrar os arquivos CSV em: {PASTA_DRIVE}")
    print("Verifique se os nomes e o caminho estão corretos.")
    exit()

# Verifica se os dataframes estão vazios após a limpeza
if df_mata.empty or df_casa.empty:
    print("\nERRO: Um dos DataFrames está vazio após o carregamento e limpeza.")
    print("Verifique se os arquivos CSV no Drive têm dados válidos.")
    exit()

# ====================================================================
# 2. Recálculo das Médias Ponderadas (Baseado em Duração)
# ====================================================================

# A) Média Ponderada para Mata (df_mata)
duracao_mata = df_mata.index.to_series().diff().shift(-1).fillna(method='ffill')
if len(duracao_mata) > 1 and duracao_mata.iloc[-1].total_seconds() == 0:
    duracao_mata.iloc[-1] = duracao_mata.iloc[-2]
elif len(duracao_mata) == 1:
    duracao_mata.iloc[0] = pd.Timedelta(seconds=1)
pesos_mata = duracao_mata.dt.total_seconds()

# O np.average é mais robusto para cálculo ponderado
media_temp_mata = np.average(df_mata['Temperatura'], weights=pesos_mata)
media_umid_mata = np.average(df_mata['Umidade'], weights=pesos_mata)


# B) Média Ponderada para Casa (df_casa), por Dispositivo
resultados_casa = []
grupos_dispositivo = df_casa.groupby('Dispositivo')

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue # Ignora grupos vazios

    duracao_casa = df_grupo.index.to_series().diff().shift(-1).fillna(method='ffill')

    if len(duracao_casa) > 1 and duracao_casa.iloc[-1].total_seconds() == 0:
        duracao_casa.iloc[-1] = duracao_casa.iloc[-2]
    elif len(duracao_casa) == 1:
        duracao_casa.iloc[0] = pd.Timedelta(seconds=1)

    pesos_casa = duracao_casa.dt.total_seconds()

    media_temp_casa = np.average(df_grupo['Temp'], weights=pesos_casa)
    media_umid_casa = np.average(df_grupo['Umid'], weights=pesos_casa)

    resultados_casa.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Ponderada': media_temp_casa,
        'Umid_Média_Ponderada': media_umid_casa
    })

df_resultados_casa = pd.DataFrame(resultados_casa).set_index('Dispositivo')


# ====================================================================
# 3. Comparação, Tabela e Conclusões
# ====================================================================

# Criar as colunas de comparação
df_resultados_casa['Diferença_Temp_vs_Mata (°C)'] = df_resultados_casa['Temp_Média_Ponderada'] - media_temp_mata
df_resultados_casa['Diferença_Umid_vs_Mata (%)'] = df_resultados_casa['Umid_Média_Ponderada'] - media_umid_mata

# Tabela final de comparação
df_comparacao = df_resultados_casa[[
    'Temp_Média_Ponderada',
    'Diferença_Temp_vs_Mata (°C)',
    'Umid_Média_Ponderada',
    'Diferença_Umid_vs_Mata (%)'
]].round(2)

print("\n" + "="*80)
print(f"|  Média Ponderada da MATA (Baseline): Temp: {media_temp_mata:.2f}°C | Umid: {media_umid_mata:.2f}%  |")
print("="*80 + "\n")

print("--- 📊 Tabela de Comparação Casa (Dispositivo) vs. Mata (Diferença) 📊 ---")
print(df_comparacao)

# -----------------
# Análise e Conclusões
# -----------------
conclusoes = []
temp_maior_que_mata = df_comparacao['Diferença_Temp_vs_Mata (°C)'].gt(0).sum()
umid_menor_que_mata = df_comparacao['Diferença_Umid_vs_Mata (%)'].lt(0).sum()
num_dispositivos = len(df_comparacao)

if num_dispositivos > 0:
    # Conclusão 1: Tendência de Temperatura
    if temp_maior_que_mata == num_dispositivos:
        conclusoes.append("→ A Casa é, em média, **mais quente** que a Mata em todas as áreas monitoradas.")
    elif temp_maior_que_mata == 0:
        conclusoes.append("→ A Casa é, em média, **mais fria** que a Mata em todas as áreas monitoradas.")
    else:
        conclusoes.append(f"→ Há uma variação: {temp_maior_que_mata}/{num_dispositivos} dispositivos na Casa são, em média, mais quentes que a Mata, indicando **microclimas internos** distintos.")

    # Conclusão 2: Tendência de Umidade
    if umid_menor_que_mata == num_dispositivos:
        conclusoes.append("→ A Casa é consistentemente **mais seca** que a Mata em todas as áreas.")
    elif umid_menor_que_mata == 0:
        conclusoes.append("→ A Casa é consistentemente **mais húmida** ou tem a mesma humidade que a Mata.")
    else:
        conclusoes.append(f"→ A maioria dos dispositivos ({umid_menor_que_mata}/{num_dispositivos}) indica uma Casa mais seca, mas alguns pontos podem estar com **humidade elevada**.")

    # Conclusão 3: Homogeneidade Interna
    std_temp = df_comparacao['Temp_Média_Ponderada'].std()
    conclusoes.append(f"→ **Disparidade Interna (Desvio Padrão Temp: {std_temp:.2f}°C):** Uma variação superior a 1 ou 2 graus indica grandes diferenças de temperatura entre os cômodos da Casa.")

print("\n--- 🧠 Conclusões da Análise Comparativa (Casa vs. Mata) 🧠 ---")
for c in conclusoes:
    print(c)

Montando Google Drive...
Mounted at /content/drive
DataFrames carregados e tipos de dados verificados com sucesso.

|  Média Ponderada da MATA (Baseline): Temp: 35.03°C | Umid: 26.19%  |

--- 📊 Tabela de Comparação Casa (Dispositivo) vs. Mata (Diferença) 📊 ---
               Temp_Média_Ponderada  Diferença_Temp_vs_Mata (°C)  \
Dispositivo                                                        
Pico W_11f0bb                 36.39                         1.36   
Pico W_11f60f                 34.73                        -0.31   

               Umid_Média_Ponderada  Diferença_Umid_vs_Mata (%)  
Dispositivo                                                      
Pico W_11f0bb                 25.47                       -0.72  
Pico W_11f60f                 27.38                        1.19  

--- 🧠 Conclusões da Análise Comparativa (Casa vs. Mata) 🧠 ---
→ Há uma variação: 1/2 dispositivos na Casa são, em média, mais quentes que a Mata, indicando **microclimas internos** distintos.
→ A maior

/tmp/ipython-input-3246839275.py:54: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  duracao_mata = df_mata.index.to_series().diff().shift(-1).fillna(method='ffill')
/tmp/ipython-input-3246839275.py:74: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  duracao_casa = df_grupo.index.to_series().diff().shift(-1).fillna(method='ffill')
